# KOA-ORDNet: training on Google Colab

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**. Allow Google Drive access when asked: checkpoints and results are saved to
   `MyDrive/KOA-ORDNet`, so nothing is lost if Colab disconnects.

Each model takes about 1–2 hours on a T4. Free Colab sessions can stop after a few hours: if that happens,
run all again and finished models are skipped automatically.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Turn on GPU: Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/KOA-ORDNet'
import os; os.makedirs(SAVE, exist_ok=True); print('Saving to', SAVE)

In [ ]:
!rm -rf /content/KOA-ORDNet
!git clone -q https://github.com/abhay93682-lab/KOA-ORDNet /content/KOA-ORDNet
%cd /content/KOA-ORDNet
!pip install -q opencv-python-headless scikit-learn kagglehub

In [ ]:
# Download the Chen (2018) dataset from Kaggle (public, CC BY 4.0)
import kagglehub
from pathlib import Path
root = kagglehub.dataset_download('shashwatwork/knee-osteoarthritis-dataset-with-severity')
DATA = None
for p in sorted(Path(root).rglob('train')):
    if p.is_dir() and all((p / str(g)).is_dir() for g in range(5)) and (p.parent / 'test').is_dir():
        DATA = str(p.parent); break
assert DATA, f'Dataset layout not found under {root}'
print('DATA =', DATA)
for s in ('train', 'val', 'test'):
    print(s, [len(list((Path(DATA) / s / str(g)).glob('*'))) for g in range(5)])

## Train the three models (skips any model already finished in Drive)

In [ ]:
import os, subprocess
RUNS = f'{SAVE}/runs'
for loss in ['ce', 'ordinal', 'koa']:
    if os.path.exists(f'{RUNS}/{loss}/history.json'):
        print(f'== {loss}: already trained, skipping'); continue
    print(f'== training {loss}')
    !python train.py --data "$DATA" --out "$RUNS" --loss $loss --workers 2

## Evaluate: Table II, Table III and Grad-CAM IoU, from the trained checkpoints
External test set (RG3): *Digital Knee X-ray Images* (Gornale & Patravali, 2020, Mendeley Data,
doi:10.17632/t9ndx37v5h.1), KL labels from Medical Expert I. It is never used for training or tuning.

In [ ]:
ext_root = kagglehub.dataset_download('tommyngx/digital-knee-xray')
EXT = next(str(p) for p in Path(ext_root).rglob('MedicalExpert-I') if p.is_dir())
print('EXT =', EXT)
!python evaluate.py --data "$DATA" --runs "$RUNS" --external "$EXT" --workers 2
!cp -r results "$SAVE"/
!cp "$RUNS"/koa/best.pt "$SAVE"/koa_ordnet.pt
print(open('results/RESULTS.md').read())
print('Saved in Google Drive:', SAVE)